In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # comment this out if running interactively and you want inline plots
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Updated columns for indexData.csv
NUMERIC_COLS = [
    "Open",
    "High",
    "Low",
    "Close",
    "Adj Close",
    "Volume",
]
CATEGORICAL_COLS = [
    "Index",
]

print("=" * 70)
print("INDEXDATA FINANCIAL CLUSTERING: K-Means / Hierarchical / DBSCAN")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Load + preprocess
# ---------------------------------------------------------------------------
df = pd.read_csv("indexData.csv")

imputer = SimpleImputer(strategy="median")
num_df = pd.DataFrame(
    imputer.fit_transform(df[NUMERIC_COLS]), columns=NUMERIC_COLS, index=df.index
)
cat_df = pd.get_dummies(df[CATEGORICAL_COLS], drop_first=True)
feature_df = pd.concat([num_df, cat_df], axis=1)
X = StandardScaler().fit_transform(feature_df)

print(f"Loaded {len(df):,} rows, {feature_df.shape[1]} clustering features (Date excluded)")

# ---------------------------------------------------------------------------
# 2. Choose k for K-Means via elbow + silhouette
# ---------------------------------------------------------------------------
k_range = range(2, 9)
sil_sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), min(5000, len(X)), replace=False)
X_sil_sample = X[sil_sample_idx]

inertias, sil_scores = [], []
for k in k_range:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_k = km_k.fit_predict(X)
    inertias.append(km_k.inertia_)
    sil_scores.append(silhouette_score(X_sil_sample, km_k.predict(X_sil_sample)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(list(k_range), inertias, marker="o")
ax1.set_title("Elbow method (inertia)")
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")

ax2.plot(list(k_range), sil_scores, marker="o", color="darkorange")
ax2.set_title("Silhouette score vs k")
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette score")

plt.tight_layout()
plt.savefig("kmeans_elbow_silhouette.png", dpi=150)
print("Saved plot -> kmeans_elbow_silhouette.png")
plt.close(fig)

best_k = list(k_range)[int(np.argmax(sil_scores))]
print(
    f"Silhouette-suggested k = {best_k}  (scores: "
    + ", ".join(f"k={k}: {s:.3f}" for k, s in zip(k_range, sil_scores))
    + ")"
)

# ---------------------------------------------------------------------------
# 3. K-Means on the FULL dataset
# ---------------------------------------------------------------------------
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
km_labels = kmeans.fit_predict(X)
sil = silhouette_score(X[sil_sample_idx], km_labels[sil_sample_idx])

print(f"\nK-Means (k={best_k}) silhouette (sample): {sil:.3f}")
print("Cluster sizes:", pd.Series(km_labels).value_counts().sort_index().to_dict())

# --- Profile clusters ---
profiled = feature_df.copy()
profiled["KMeansCluster"] = km_labels
profiled["IndexType"] = df["Index"].values
summary = (
    profiled.groupby("KMeansCluster")
    .agg(
        n_rows=("KMeansCluster", "size"),
        avg_Open=("Open", "mean"),
        avg_Close=("Close", "mean"),
        avg_Volume=("Volume", "mean"),
    )
    .round(3)
)
print("\n--- Cluster profiles (KMeansCluster) ---")
print(summary.to_string())
summary.to_csv("cluster_profiles.csv")
print("Saved table -> cluster_profiles.csv")

# ---------------------------------------------------------------------------
# 4. Hierarchical + DBSCAN on a subsample
# ---------------------------------------------------------------------------
sub_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), min(5000, len(X)), replace=False)
X_sub = X[sub_idx]

hc = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
hc_labels = hc.fit_predict(X_sub)
sil_hc = silhouette_score(X_sub, hc_labels)
print(f"\nHierarchical (k={best_k}, subsample) silhouette: {sil_hc:.3f}")

db = DBSCAN(eps=4.0, min_samples=15)
db_labels = db.fit_predict(X_sub)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = int(np.sum(db_labels == -1))
print(f"DBSCAN (subsample): {n_db_clusters} clusters, {n_noise} noise points")

if n_db_clusters >= 2:
    mask = db_labels != -1
    print(f"DBSCAN silhouette (excl. noise): {silhouette_score(X_sub[mask], db_labels[mask]):.3f}")

# --- Dendrogram ---
dendro_idx = np.random.choice(len(X_sub), size=min(80, len(X_sub)), replace=False)
Z = linkage(X_sub[dendro_idx], method="ward")
plt.figure(figsize=(12, 5))
dendrogram(Z)
plt.title("Hierarchical Clustering Dendrogram (Ward linkage, subsample)")
plt.xlabel("Index row")
plt.ylabel("Distance")
plt.tight_layout()
plt.savefig("dendrogram.png", dpi=150)
print("Saved plot -> dendrogram.png")
plt.close()

# ---------------------------------------------------------------------------
# 5. PCA for visualization
# ---------------------------------------------------------------------------
X_pca_full = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)

fig, ax = plt.subplots(figsize=(6, 5.2))
ax.scatter(X_pca_full[:, 0], X_pca_full[:, 1], c=km_labels, cmap="tab10", s=8, alpha=0.6)
ax.set_title(f"K-Means, full data ({best_k} clusters)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.savefig("kmeans_pca.png", dpi=150)
print("Saved plot -> kmeans_pca.png")
plt.close(fig)

# ---------------------------------------------------------------------------
# 6. Save deliverable: original data + cluster assignment
# ---------------------------------------------------------------------------
df_out = df.copy()
df_out["KMeansCluster"] = km_labels
df_out.to_csv("indexData_with_clusters.csv", index=False)
print("\nSaved deliverable -> indexData_with_clusters.csv")
print("\nDone. Clustering complete for indexData!")

INDEXDATA FINANCIAL CLUSTERING: K-Means / Hierarchical / DBSCAN
Loaded 112,457 rows, 19 clustering features (Date excluded)
Saved plot -> kmeans_elbow_silhouette.png
Silhouette-suggested k = 2  (scores: k=2: 0.617, k=3: 0.221, k=4: 0.234, k=5: 0.247, k=6: 0.274, k=7: 0.352, k=8: 0.402)

K-Means (k=2) silhouette (sample): 0.617
Cluster sizes: {0: 110076, 1: 2381}

--- Cluster profiles (KMeansCluster) ---
               n_rows   avg_Open  avg_Close    avg_Volume
KMeansCluster                                            
0              110076   6678.004   6676.745  1.276032e+09
1                2381  50707.280  50721.209  0.000000e+00
Saved table -> cluster_profiles.csv

Hierarchical (k=2, subsample) silhouette: 0.617
DBSCAN (subsample): 7 clusters, 1 noise points
DBSCAN silhouette (excl. noise): 0.355
Saved plot -> dendrogram.png
Saved plot -> kmeans_pca.png

Saved deliverable -> indexData_with_clusters.csv

Done. Clustering complete for indexData!
